# Evaluating generated images with automatic measure

In [ ]:
import sys
import os
sys.path.append("/path/to/repo/")
import t2v_metrics.t2v_metrics as t2v_scores
import torch
from tqdm import tqdm
import json
import re

In [ ]:
cache_directory = "/path/to/cache/" 

In [ ]:
# load compounds and constituents
compounds = set()
constituents = set()
with open("/path/to/repo/Data/compounds.csv","r") as infile:
    for line in infile:
        if not line.startswith("ID"):
            compound = line.split(",")[1]
            compounds.add(compound.strip())
            for c in compound.split(" "):
                constituents.add(c.strip())
print(len(compounds),compounds)
print(len(constituents),constituents)

200 {'library card', 'ruling party', 'emotion processing', 'coffee machine', 'ballet shoe', 'image processing', 'emergency service', 'prom queen', 'ego trip', 'identity card', 'camping area', 'arcade game', 'potato cake', 'music critic', 'war movie', 'family romance', 'music fan', 'spirit photography', 'data quality', 'brain food', 'baby shoe', 'propaganda machine', 'farewell dinner', 'horror movie', 'confidentiality agreement', 'glove box', 'emergency equipment', 'field trip', 'bank card', 'school dinner', 'field day', 'folk music', 'dining area', 'elimination game', 'peace conference', 'suggestion box', 'space helmet', 'family dog', 'service station', 'road closure', 'culture critic', 'identity paper', 'language barrier', 'ice queen', 'fantasy romance', 'guilt trip', 'paper loss', 'gambling loss', 'child protection', 'bus station', 'service dog', 'gala dinner', 'tree surgery', 'horse shoe', 'language area', 'tennis shoe', 'ticket barrier', 'plea agreement', 'steel helmet', 'soul musi

In [ ]:
# load pretrained model
device = torch.device("cuda:"+str(2) if torch.cuda.is_available() else "cpu") # TODO choose available GPU
print(device)
vqa_score = t2v_scores.VQAScore(model="clip-flant5-xxl",cache_dir=cache_directory,device=device)

In [ ]:
evaluation_definition_template = "{target_definition} Does this figure show {target}?"
evaluation_simple_template = "Does this figure show {target}?"

In [8]:
# load the prompts
def load_prompts(filepath: str) -> dict:
    """
    Loads prompts listed in file into a dictionary.
    Parameter:
        filepath (str): link to where file is stored, lines in file are in the format "target \t prompt (\t prompt)", and for each target a separate line; at least one prompt, but multiple ones are possible
    Returns:
        prompts (dict): dictionary with targets as keys and prompts (list) as values
    """
    prompts = dict()
    with open(filepath,"r") as infile:
        for line in infile:
            prompt_info = line.split("\t")
            target = prompt_info[0]
            target_prompts = prompt_info[1:]
            if target in prompts:
                print("check the file again for the target '"+target+"', as it occurs multiple times")
            prompts[target] = target_prompts
    return prompts

### Constituent images from PixArtSigma ChatGPT-definition

In [ ]:
link_to_prompts = "/path/to/repo/Data/NounDefinitions/chatgpt_constituent_definition_prompts.tsv"
constituent_chatgpt_definitions = load_prompts(link_to_prompts)

image_dir = "/path/to/repo/Data/Images/ConstituentImages/pixartsigma/chatgpt_constituent_definitions/"
evaluation_dir = "/path/to/repo/Data/Images/ImageEvaluations/"

In [ ]:
score = vqa_score 
score_id = "VQAscores"

scores_chatgpt_constituent_definition = dict() # key = absolute image path, value = score

generated_image_paths = os.listdir(image_dir)
for generated_image_path in tqdm(generated_image_paths):
    constituent = generated_image_path.split("_")[1]
    constituent_definition = constituent_chatgpt_definitions[constituent][0].strip() # take the first definition only!
    evaluation_definition_question = evaluation_definition_template.format(target=constituent,target_definition=constituent_definition)
    evaluation_simple_question = evaluation_simple_template.format(target=constituent)
    absolute_image_path = image_dir+generated_image_path
    scores = score(images=[absolute_image_path],texts=[evaluation_simple_question,evaluation_definition_question])
    scores_chatgpt_constituent_definition[absolute_image_path] = {"simple":scores[0][0].item(),"definition":scores[0][1].item()}

# save scores to file
with open(evaluation_dir+score_id+"_constituent_images_pixartsigma_chatGPTdefinitions.json","w") as saver_file:
        json.dump(scores_chatgpt_constituent_definition,saver_file,indent=2)

### Evaluate all generated compound images

In [ ]:
# similarly as for constituent images (cf. above), now just for other image paths!